<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/compute_influence_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

In [ ]:
!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
%cd mask-predict

In [ ]:
!pip install -e .

In [ ]:
from explain import *
import pandas as pd

In [ ]:
collection = 'hmd'
maskedToken = 'machine'
clusterName = 'slave'

In [ ]:
!unzip -o "{collection}_{maskedToken}_clusters.tsv.zip"

In [ ]:
data_path = '.'
data_df = pd.read_csv(f"{data_path}/{collection}_{maskedToken}_clusters.tsv", sep="\t", index_col=0))
data_df = data_df.sort_values(by=f"{clusterName}_1760_1900", ascending=False).reset_index(drop=True)

data_df.head()

In [ ]:
iloc_range = (0, 30)  # Adjust as needed
texts, targets = build_texts_targets(
    data_df, start=iloc_range[0], end=iloc_range[1],
    pred_col="pred_bert_1760_1900", top_n=5
)


In [ ]:

explainer = MaskedLMExplainer(model_name="Livingwithmachines/bert_1760_1900", device=pick_device())
results = explainer.explain(texts, targets, word_agg="max")


In [ ]:
df = result_as_dataframe(results, ['slave', 'machine'])
df[df['Target'] == 'slave'].groupby('Token'
                                    ).agg(avg=('Score','mean'), count=('Score', 'count')
                                          ).sort_values(by='avg', ascending=False)
                                                                                                             

In [ ]:

explainer_new = MaskedLMExplainer("Livingwithmachines/bert_1760_1900", device=pick_device())
explainer_old = MaskedLMExplainer("Livingwithmachines/bert_1760_1850", device=pick_device())



In [ ]:
single_target = [["slave"] for _ in texts]

comparison = compare_explainers(explainer_old,explainer_new, texts, single_target, level="word", word_agg="max")

In [ ]:
top_predictors = summarize_top_predictors(results, target="slave", top_n=15)
print(f"Top predictors for 'slave':\n")
print(f"{'Word':>15} {'Mean':>10} {'Std':>10} {'Count':>8}")
print("-" * 45)
for word, mean, std, count in top_predictors:
    print(f"{word:>15} {mean:>10.4f} {std:>10.4f} {count:>8}")

In [ ]:
comparison_stats = analyze_comparison(comparison, target="slave", top_n=15)
print(f"Top diverging words between models for 'slave':\n")
print(f"{'Word':>15} {'Model1':>12} {'Model2':>12} {'Diff':>12} {'Std':>10}")
print("-" * 62)
for word, m1, m2, diff, std in comparison_stats:
    print(f"{word:>15} {m1:>12.4f} {m2:>12.4f} {diff:>12.4f} {std:>10.4f}")

In [ ]:
sentence = data_df.iloc[0]['maskedSentence']
highlight_context_tokens(explainer_new, sentence, target="slave", word_agg="max")


In [ ]:
plot_model_comparison_bar(comparison, target="slave", top_n=15)

plot_scatter_model_comparison(comparison, target="slave", top_n=25)

In [ ]:
render_top_shift_sentences(
    texts=texts,
    comparison=comparison,
    target="slave",
    top_k=5,
    score_mode="mean_abs",
    show=True
)


# Fin.